#Zundamotion

## VOICEVOX設定

In [ ]:
# @title VOICEVOX ENGINE設定
# @markdown [VOICEVOX ENGINE リリースページ](https://github.com/VOICEVOX/voicevox_engine/releases/)
import os
import tensorflow as tf

os.environ['ENGINE_VERSION'] = "0.25.1" # @param {type:"string"}


if tf.test.gpu_device_name():
  # GPU
  !wget "https://github.com/VOICEVOX/voicevox_engine/releases/download/${ENGINE_VERSION}/voicevox_engine-linux-nvidia-${ENGINE_VERSION}.7z.001"
  !wget "https://github.com/VOICEVOX/voicevox_engine/releases/download/${ENGINE_VERSION}/voicevox_engine-linux-nvidia-${ENGINE_VERSION}.7z.002"
  !7za x -y voicevox_engine-linux-nvidia-${ENGINE_VERSION}.7z.001
else:
  # CPU
  !wget "https://github.com/VOICEVOX/voicevox_engine/releases/download/${ENGINE_VERSION}/voicevox_engine-linux-cpu-x64-${ENGINE_VERSION}.7z.001" -o "voicevox_engine.7z"
  !7za x -y voicevox_engine-linux-cpu-${ENGINE_VERSION}.7z.001

In [ ]:
# @title VOICEVOX ENGINE 起動
# VOICEVOX ENGINE バックグランド起動
import tensorflow as tf
if tf.test.gpu_device_name():
  # GPU
  !./linux-nvidia/run --use_gps --allow_origin '*' --cors_policy_mode 'all' --host '127.0.0.1' > /dev/null 2>&1 &
else:
  # CPU
  !./linux-cpu/run --allow_origin '*' --cors_policy_mode 'all' --host '127.0.0.1' > /dev/null 2>&1 &

## Zundamotion

In [ ]:
# @title 環境構築
# 環境構築
!git clone https://github.com/c-a-p-engineer/zundamotion.git
%cd zundamotion
!python -m pip install --upgrade pip setuptools wheel
!pip install -e .
!pip install -r requirements.txt
# 日本語フォント導入
!apt-get update && apt-get install -y fonts-ipafont-gothic && rm -rf /var/lib/apt/lists/*

In [ ]:
# @title ffmpeg, ffprobe導入
!curl -L https://johnvansickle.com/ffmpeg/releases/ffmpeg-release-amd64-static.tar.xz -o ffmpeg.tar.xz
!tar -xJf ffmpeg.tar.xz
!cp ffmpeg-*-amd64-static/ffmpeg ffmpeg-*-amd64-static/ffprobe /usr/local/bin/
!chmod +x /usr/local/bin/ffmpeg /usr/local/bin/ffprobe

!ffmpeg -version

In [ ]:
# @title 動画生成
!cd /content/zundamotion
file_path = "/content/zundamotion/scripts/sample.yaml" # @param {type:"string"}
!python -m zundamotion.main {file_path}

In [ ]:
# @title 最新の生成動画表示
import os
import glob
from IPython.display import HTML
import base64

output_dir = "/content/zundamotion/output"
mp4_files = glob.glob(os.path.join(output_dir, "*.mp4"))

if mp4_files:
    latest_mp4 = max(mp4_files, key=os.path.getmtime)
    print(f"Latest MP4 file: {latest_mp4}")
else:
    print(f"No MP4 files found in {output_dir}")
    exit(1)

with open(latest_mp4, "rb") as f:
    data = f.read()
    b64 = base64.b64encode(data).decode()

html = f"""
<video width="720" controls autoplay loop muted playsinline>
  <source src="data:video/mp4;base64,{b64}" type="video/mp4">
</video>
"""

display(HTML(html))